
# IMPORT LIBRARIES


In [11]:

import pandas as pd
from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel, logging
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# path to dataset folder
DATASET_DIR = Path("dataset")

# HELPER FUNCTIONS

In [12]:
# simple text cleaning (no regex)
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())  # removes extra spaces
    return text.strip()

# show dataset info
def show(df, name):
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Label counts:")
    print(df["label"].value_counts().sort_index())
    display(df.head())

# undersample if imbalance is high
def undersample(df, name, threshold=1.1):
    counts = df["label"].value_counts()

    # if only one class exists
    if len(counts) < 2:
        print(f"\n{name}: only one class found, undersampling skipped")
        return df.reset_index(drop=True)

    ratio = counts.max() / counts.min()
    print(f"\n{name} imbalance ratio: {ratio:.2f}")

    # already balanced
    if ratio <= threshold:
        print(f"{name}: no undersampling needed")
        return df.reset_index(drop=True)

    print(f"{name}: undersampling applied")

    major_class = counts.idxmax()
    minor_class = counts.idxmin()

    major_df = df[df["label"] == major_class].sample(n=counts.min(), random_state=42)
    minor_df = df[df["label"] == minor_class]

    balanced_df = pd.concat([major_df, minor_df], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df


# LOAD AND PREPARE ISOT DATASET

In [13]:
def load_isot():
    fake = pd.read_csv(DATASET_DIR / "ISOT_Fake_News Dataset" / "Fake.csv")
    real = pd.read_csv(DATASET_DIR / "ISOT_Fake_News Dataset" / "True.csv")

    fake["label"] = 1
    real["label"] = 0

    df = pd.concat([fake, real], ignore_index=True)

    df["content"] = (df["title"].astype(str) + " " + df["text"].astype(str)).apply(clean_text)
    df = df[["content", "label"]]
    df = df[df["content"].str.len() > 0].drop_duplicates().reset_index(drop=True)

    show(df, "ISOT Original")
    df = undersample(df, "ISOT")
    show(df, "ISOT Final")

    return df


# LOAD AND PREPARE FAKENEWSNET DATASET


In [14]:
def load_fnn():
    path = DATASET_DIR / "FakeNewsNet"

    gc_fake = pd.read_csv(path / "gossipcop_fake.csv")
    gc_real = pd.read_csv(path / "gossipcop_real.csv")
    pf_fake = pd.read_csv(path / "politifact_fake.csv")
    pf_real = pd.read_csv(path / "politifact_real.csv")

    gc_fake["label"] = 1
    gc_real["label"] = 0
    pf_fake["label"] = 1
    pf_real["label"] = 0

    df = pd.concat([gc_fake, gc_real, pf_fake, pf_real], ignore_index=True)

    # find text columns
    title_col = "title" if "title" in df.columns else None
    text_col = None
    for col in ["text", "content", "body"]:
        if col in df.columns:
            text_col = col
            break

    # build content
    if title_col and text_col:
        df["content"] = df[title_col].astype(str) + " " + df[text_col].astype(str)
    elif text_col:
        df["content"] = df[text_col].astype(str)
    elif title_col:
        df["content"] = df[title_col].astype(str)
    else:
        raise ValueError("No usable text column found")

    df["content"] = df["content"].apply(clean_text)
    df = df[["content", "label"]]
    df = df[df["content"].str.len() > 0].drop_duplicates().reset_index(drop=True)

    show(df, "FakeNewsNet Original")
    df = undersample(df, "FakeNewsNet")
    show(df, "FakeNewsNet Final")

    return df

# LOAD AND PREPARE LIAR DATASET

In [15]:
def load_liar():
    path = DATASET_DIR / "LIAR_PLUS" / "tsv"

    cols = [
        "id", "json", "label", "statement", "topic", "speaker",
        "job_title", "state_info", "party_affiliation",
        "barely_true_counts", "false_counts", "half_true_counts",
        "mostly_true_counts", "pants_on_fire_counts",
        "context", "justification"
    ]

    train = pd.read_csv(path / "train2.tsv", sep="\t", header=None, names=cols)
    val = pd.read_csv(path / "val2.tsv", sep="\t", header=None, names=cols)
    test = pd.read_csv(path / "test2.tsv", sep="\t", header=None, names=cols)

    df = pd.concat([train, val, test], ignore_index=True)

    label_map = {
        "true": 0,
        "mostly-true": 0,
        "half-true": 0,
        "false": 1,
        "barely-true": 1,
        "pants-fire": 1
    }

    df["label"] = df["label"].map(label_map)
    df["content"] = df["statement"].apply(clean_text)

    df = df[["content", "label"]].dropna()
    df = df[df["content"].str.len() > 0].drop_duplicates().reset_index(drop=True)
    df["label"] = df["label"].astype(int)

    show(df, "LIAR Original")
    df = undersample(df, "LIAR")
    show(df, "LIAR Final")

    return df

# LOAD ALL DATASETS

In [16]:

isot = load_isot()


ISOT Original
Shape: (39100, 2)
Label counts:
label
0    21195
1    17905
Name: count, dtype: int64


,content,label
0,Donald Trump Sends Out Embarrassing New Year’s...,1
1,Drunk Bragging Trump Staffer Started Russian C...,1
2,Sheriff David Clarke Becomes An Internet Joke ...,1
3,Trump Is So Obsessed He Even Has Obama’s Name ...,1
4,Pope Francis Just Called Out Donald Trump Duri...,1



ISOT imbalance ratio: 1.18
ISOT: undersampling applied

ISOT Final
Shape: (35810, 2)
Label counts:
label
0    17905
1    17905
Name: count, dtype: int64


,content,label
0,ALARMING! THIS STATE’S SCHOOL SYSTEM TO TAKE O...,1
1,"BREAKING UPDATE: 50 DEAD, 53 Injured After U.S...",1
2,MAJOR COSMETIC Company Announces Plans To Rele...,1
3,3 CONSERVATIVE CELEBRITIES Visited Trump Last ...,1
4,Fox News says Google to partner in Jan. 28 Iow...,0


In [17]:
fnn = load_fnn()


FakeNewsNet Original
Shape: (21847, 2)
Label counts:
label
0    16524
1     5323
Name: count, dtype: int64


,content,label
0,Did Miley Cyrus and Liam Hemsworth secretly ge...,1
1,Paris Jackson & Cara Delevingne Enjoy Night Ou...,1
2,Celebrities Join Tax March in Protest of Donal...,1
3,Cindy Crawford's daughter Kaia Gerber wears a ...,1
4,Full List of 2018 Oscar Nominations – Variety,1



FakeNewsNet imbalance ratio: 3.10
FakeNewsNet: undersampling applied

FakeNewsNet Final
Shape: (10646, 2)
Label counts:
label
0    5323
1    5323
Name: count, dtype: int64


,content,label
0,Anna Faris Feels Like “Leaving The Country” Af...,0
1,Kim and Khloe Kardashian Talk Blac Chyna and R...,0
2,Kim Kardashian and Kanye West Name Baby No. 3 ...,1
3,V Magazine,0
4,Taylor Swift ‎– Greatest Hits,1


In [18]:
liar = load_liar()


LIAR Original
Shape: (12769, 2)
Label counts:
label
0    7125
1    5644
Name: count, dtype: int64


,content,label
0,Says the Annies List political group supports ...,1
1,When did the decline of coal start? It started...,0
2,"Hillary Clinton agrees with John McCain ""by vo...",0
3,Health care reform legislation is likely to ma...,1
4,The economic turnaround started at the end of ...,0



LIAR imbalance ratio: 1.26
LIAR: undersampling applied

LIAR Final
Shape: (11288, 2)
Label counts:
label
0    5644
1    5644
Name: count, dtype: int64


,content,label
0,Toomey crossed party lines to stop gun sales t...,1
1,Large majorities of the public oppose major ch...,0
2,Says the fluoride Austin is putting in its dri...,1
3,"As unbelievable as it sounds, your tax dollars...",1
4,The U.S. has the highest unintended pregnancy ...,0


# SHOW FINAL DATASET SAMPLE COUNTS

In [19]:

summary = pd.DataFrame({
    "Dataset": ["ISOT", "FakeNewsNet", "LIAR"],
    "Samples": [len(isot), len(fnn), len(liar)],
    "Real (0)": [
        isot["label"].value_counts().get(0, 0),
        fnn["label"].value_counts().get(0, 0),
        liar["label"].value_counts().get(0, 0)
    ],
    "Fake (1)": [
        isot["label"].value_counts().get(1, 0),
        fnn["label"].value_counts().get(1, 0),
        liar["label"].value_counts().get(1, 0)
    ]
})

print("\nFinal Dataset Sizes")
display(summary)


Final Dataset Sizes


,Dataset,Samples,Real (0),Fake (1)
0,ISOT,35810,17905,17905
1,FakeNewsNet,10646,5323,5323
2,LIAR,11288,5644,5644



# TUNABLE PARAMETERS

In [20]:



EPOCHS = 10

# bigger batch = faster
BATCH_SIZE = 32

# shorter sequence = much faster
MAX_LEN = 64

LEARNING_RATE = 5e-5

# slightly lower dropout for faster convergence
DROPOUT = 0.2

TEST_SIZE = 0.2
VAL_SIZE = 0.1

RANDOM_STATE = 42

# None = full dataset
SAMPLE_SIZE = 5000

# early stopping
EARLY_STOPPING_PATIENCE = 2

TRAIN_ISOT = True
TRAIN_FNN = True
TRAIN_LIAR = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-uncased"
)

Device: cpu


# DATASET CLASS

In [21]:
class FakeNewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len):

        self.labels = labels.tolist()

        self.encodings = tokenizer(
            texts.tolist(),
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "label": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

# MODEL


In [22]:
# =========================
# MODEL
# =========================

class BertFakeNewsClassifier(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = BertModel.from_pretrained(
            "bert-base-uncased"
        )

        self.dropout = torch.nn.Dropout(DROPOUT)

        self.classifier = torch.nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.pooler_output

        pooled_output = self.dropout(
            pooled_output
        )

        logits = self.classifier(
            pooled_output
        )

        return logits

# TRAIN FUNCTION

In [23]:
# =========================
# TRAIN FUNCTION
# =========================

def train_one_epoch(
    model,
    loader,
    optimizer,
    loss_fn
):

    model.train()

    total_loss = 0

    preds = []
    labels = []

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask
        )

        loss = loss_fn(outputs, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        batch_preds = torch.argmax(
            outputs,
            dim=1
        )

        preds.extend(
            batch_preds.cpu().numpy()
        )

        labels.extend(
            y.cpu().numpy()
        )

    acc = accuracy_score(
        labels,
        preds
    )

    return total_loss / len(loader), acc

# EVALUATION

In [24]:


# =========================
# EVALUATION
# =========================

def evaluate(
    model,
    loader,
    loss_fn
):

    model.eval()

    total_loss = 0

    preds = []
    labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)

            outputs = model(
                input_ids,
                attention_mask
            )

            loss = loss_fn(outputs, y)

            total_loss += loss.item()

            batch_preds = torch.argmax(
                outputs,
                dim=1
            )

            preds.extend(
                batch_preds.cpu().numpy()
            )

            labels.extend(
                y.cpu().numpy()
            )

    acc = accuracy_score(
        labels,
        preds
    )

    return total_loss / len(loader), acc

# MAIN TRAINING

In [25]:

# =========================
# MAIN TRAINING
# =========================

def train_bert(
    df,
    dataset_name
):

    print(f"\n========================")
    print(f"Training {dataset_name}")
    print(f"========================")

    df = df.copy().reset_index(drop=True)

    if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:

        df = df.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)

    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        stratify=df["label"],
        random_state=RANDOM_STATE
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        stratify=train_df["label"],
        random_state=RANDOM_STATE
    )

    train_dataset = FakeNewsDataset(
        train_df["content"],
        train_df["label"],
        tokenizer,
        MAX_LEN
    )

    val_dataset = FakeNewsDataset(
        val_df["content"],
        val_df["label"],
        tokenizer,
        MAX_LEN
    )

    test_dataset = FakeNewsDataset(
        test_df["content"],
        test_df["label"],
        tokenizer,
        MAX_LEN
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    model = BertFakeNewsClassifier().to(device)

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    loss_fn = torch.nn.CrossEntropyLoss()

    best_val_acc = 0

    patience_counter = 0

    best_model_path = f"bert_{dataset_name}_best.pt"

    for epoch in range(EPOCHS):

        print(f"\nEpoch {epoch + 1}/{EPOCHS}")

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn
        )

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            loss_fn
        )

        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Validation Accuracy: {val_acc:.4f}")

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            patience_counter = 0

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Best model saved")

        else:

            patience_counter += 1

            print(
                f"No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})"
            )

            if patience_counter >= EARLY_STOPPING_PATIENCE:

                print("Early stopping triggered")

                break

    # =========================
    # TEST
    # =========================

    model.load_state_dict(
        torch.load(best_model_path)
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        loss_fn
    )

    print(f"Test Accuracy: {test_acc:.4f}")

    return {
        "Dataset": dataset_name,
        "Samples Used": len(df),
        "Best Validation Accuracy": round(best_val_acc, 4),
        "Test Accuracy": round(test_acc, 4)
    }


# RUN TRAINING

In [ ]:

# =========================
# RUN TRAINING
# =========================

results = []

if TRAIN_ISOT:
    results.append(
        train_bert(isot, "ISOT")
    )

if TRAIN_FNN:
    results.append(
        train_bert(fnn, "FakeNewsNet")
    )

if TRAIN_LIAR:
    results.append(
        train_bert(liar, "LIAR")
    )


Training ISOT


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21036.00it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/10


# FINAL RESULTS

In [ ]:


# =========================
# FINAL RESULTS
# =========================

results_df = pd.DataFrame(results)

print("\nFinal Results")

display(results_df)

Device: cpu

Training ISOT


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20636.56it/s]



Epoch 1/10
Train Accuracy: 0.9883
Validation Accuracy: 1.0000
Best model saved

Epoch 2/10
Train Accuracy: 0.9986
Validation Accuracy: 1.0000
No improvement (1/2)

Epoch 3/10
Train Accuracy: 0.9994
Validation Accuracy: 1.0000
No improvement (2/2)
Early stopping triggered
Test Accuracy: 0.9990

Training FakeNewsNet


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8964.40it/s]



Epoch 1/10
Train Accuracy: 0.6883
Validation Accuracy: 0.8050
Best model saved

Epoch 2/10
